<a href="https://colab.research.google.com/github/romes-dev/pipeline-ceub/blob/main/Melhorando_com_tfkeras_TM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🐱🐶 Teachable Machine → Colab (TF 2.19 + `tf-keras`)

Notebook completo para **avaliar** e **fazer fine-tuning** de um modelo de *cats vs dogs* exportado do **Teachable Machine**.

Este template assume o runtime padrão do Colab (TensorFlow 2.19 + Keras 3). Para carregar `.h5` do TM, usamos **`tf-keras`** (ponte compatível com modelos Keras 2.x).

**Estrutura esperada em `/content`**:
```
/content/
  keras_model.h5
  labels.txt
  test/
    cat/...
    dog/...
  data/  (opcional, p/ fine-tuning)
    cat/...
    dog/...
```


## 1) Instalação e checagem de versões

In [ ]:
!pip -q install "tf-keras==2.19.0" "scikit-learn>=1.3" pillow matplotlib
import tensorflow as tf
import tf_keras as keras
print("TensorFlow:", tf.__version__)
print("tf-keras:", keras.__version__)

## 2) Upload rápido (se precisar)

In [ ]:
from google.colab import files
print("Selecione `keras_model.h5` e `labels.txt` se ainda não estiverem no runtime...")
files.upload()

## 3) Carregar modelo `.h5` do Teachable Machine e rótulos

In [ ]:
from tf_keras.models import load_model
from pathlib import Path

assert Path("keras_model.h5").exists(), "Coloque `keras_model.h5` em /content"
assert Path("labels.txt").exists(), "Coloque `labels.txt` em /content"

model = load_model("keras_model.h5", compile=False)
with open("labels.txt") as f:
    class_names = [ln.strip().split()[-1] for ln in f if ln.strip()]

IMG_SIZE = (224, 224)  # MobileNet (padrão do TM)
def preprocess(x):
    x = tf.image.resize(x, IMG_SIZE)
    x = tf.cast(x, tf.float32)
    return (x / 127.5) - 1.0  # normalização usada pelo TM

print("Classes (labels.txt):", class_names)

## 4) Avaliação no conjunto de **teste** (separado do TM)

In [ ]:
import numpy as np
import tensorflow as tf
from pathlib import Path

assert Path("test").exists(), "Crie a pasta `test/` com subpastas por classe (ex.: cat/, dog/ ou gato/, cachorro/)"

# 1) Carregue cru, pegue as classes e só depois faça o map/prefetch
raw_test_ds = tf.keras.utils.image_dataset_from_directory(
    "test", image_size=IMG_SIZE, batch_size=32, shuffle=False
)
ds_class_names = raw_test_ds.class_names  # <-- salve aqui!
print("Classes detectadas no test/:", ds_class_names)

test_ds = raw_test_ds.map(lambda x,y: (preprocess(x), y)).prefetch(tf.data.AUTOTUNE)

y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_prob = model.predict(test_ds)
y_pred = y_prob.argmax(axis=1)
print("Predições concluídas.")


## 5) Métricas: relatório, matriz de confusão e ROC/AUC

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay
import matplotlib.pyplot as plt

print(classification_report(y_true, y_pred, target_names=ds_class_names))

cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=ds_class_names).plot(xticks_rotation=45); plt.show()

# ROC/AUC (binário)
if len(ds_class_names) == 2:
    # tenta achar automaticamente o índice da classe positiva
    possible_pos = ["dog", "cachorro"]
    pos_index = next((ds_class_names.index(n) for n in possible_pos if n in ds_class_names), 1)
    y_pos = (y_true == pos_index).astype(int)
    print("AUC:", roc_auc_score(y_pos, y_prob[:, pos_index]))
    RocCurveDisplay.from_predictions(y_pos, y_prob[:, pos_index]); plt.show()
else:
    print("ROC/AUC omitido (não binário).")


## 6) Ajuste de *threshold* para otimizar F1/Recall/Precisão (binário)

In [ ]:
from sklearn.metrics import precision_recall_curve
import numpy as np

if len(class_names) == 2:
    pos_index = 1
    y_pos = (y_true==pos_index).astype(int)
    prec, rec, thr = precision_recall_curve(y_pos, y_prob[:, pos_index])
    f1 = 2*prec*rec/(prec+rec+1e-8)
    best_idx = np.nanargmax(f1[:-1])
    best_thr = float(thr[best_idx])
    print(f"Melhor threshold (F1): {best_thr:.3f} | P={prec[best_idx]:.3f}, R={rec[best_idx]:.3f}")
else:
    print("Ajuste de limiar é relevante para binário.")

## 7) Fine-tuning leve (opcional)

Use a pasta `data/` com **mais imagens** para continuar treinando. O código descongela as últimas camadas e aplica *data augmentation*. Se o dataset for desbalanceado, calculamos `class_weight` automaticamente.


In [ ]:
import collections
from pathlib import Path
from tf_keras import layers, optimizers, callbacks

if Path("data").exists():
    train_ds = tf.keras.utils.image_dataset_from_directory(
        "data", validation_split=0.2, subset="training", seed=1337,
        image_size=IMG_SIZE, batch_size=32)
    val_ds = tf.keras.utils.image_dataset_from_directory(
        "data", validation_split=0.2, subset="validation", seed=1337,
        image_size=IMG_SIZE, batch_size=32)

    data_aug = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.1),
    ])

    train_ds = train_ds.map(lambda x,y: (preprocess(data_aug(x)), y)).prefetch(tf.data.AUTOTUNE)
    val_ds   = val_ds.map(lambda x,y: (preprocess(x), y)).prefetch(tf.data.AUTOTUNE)

    # Descongele uma parte final do modelo (ajuste o número de camadas conforme necessidade)
    for layer in model.layers: layer.trainable = False
    for layer in model.layers[-20:]: layer.trainable = True

    # Pesos por classe (se houver desbalanceamento)
    counts = collections.Counter(np.concatenate([y.numpy() for _,y in train_ds]))
    total = sum(counts.values())
    num_classes = len(counts)
    class_weight = {i: total/(num_classes*counts[i]) for i in counts}
    print("class_weight:", class_weight)

    model.compile(optimizer=optimizers.Adam(1e-4),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])

    cb = [callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
    history = model.fit(train_ds, validation_data=val_ds, epochs=20,
                        callbacks=cb, class_weight=class_weight)

    print("Val eval:", model.evaluate(val_ds))
    model.save("tm_catdog_finetuned.h5")
    print("Modelo salvo como tm_catdog_finetuned.h5")
else:
    print("Pasta `data/` não encontrada. Pule esta etapa se não vai treinar agora.")

## 8) Inferência em **uma imagem** (sanity check)

In [ ]:
from PIL import Image
import numpy as np
from google.colab import files

print("Envie uma imagem .jpg/.png para testar a predição...")
up = files.upload()
img_path = list(up.keys())[0]
img = Image.open(img_path).convert("RGB")
arr = np.array(img)
x = tf.expand_dims(arr, 0)
x = preprocess(x)
prob = model.predict(x)[0]
pred = int(np.argmax(prob))
print({"pred_label": class_names[pred] if pred < len(class_names) else pred,
       "probs": {class_names[i] if i < len(class_names) else str(i): float(p) for i,p in enumerate(prob)}})

## 9) (Opcional) Carregar **SavedModel** ao invés de `.h5`
Se você re-exportar do Teachable Machine como **TensorFlow (SavedModel)**, coloque a pasta em `/content/saved_model/` e use:


In [ ]:
from tf_keras.models import load_model
from pathlib import Path
if Path("saved_model").exists():
    model = load_model("saved_model")
    print("SavedModel carregado com sucesso.")
else:
    print("Pasta `saved_model/` não encontrada (ignore se usa .h5).")